In [2]:
import pandas as pd  
import glob
from tqdm.notebook import tqdm

In [ ]:
# Ruta donde están tus CSV (ej: "/Users/tu_usuario/ecobici/")
ruta_csv = "./*.csv"  # Cambia esto si tus archivos están en otra carpeta
mapeo_columnas = {
    "Genero_usuario": "Genero_Usuario",
    "Edad_usuario": "Edad_Usuario",
    "BikeID": "Bici",
    "CE_retiro":"Ciclo_Estacion_Retiro",
    "Fecha_retiro": "Fecha_Retiro",
    "Hora_retiro":"Hora_Retiro",
    "CE_arribo":"Ciclo_Estacion_Arribo",
    "Ciclo_EstacionArribo": "Ciclo_Estacion_Arribo",
    "Fecha Arribo": "Fecha_Arribo",
    "Fecha_arribo": "Fecha_Arribo",
    "Hora_arribo":"Hora_Arribo",
    "Hora Arribo":"Hora_Arribo"
}
# Procesar cada CSV
for csv_file in tqdm(glob.glob(ruta_csv), desc="Convirtiendo a Parquet"):   
    df = pd.read_csv(csv_file, engine='pyarrow').rename(columns=mapeo_columnas)  # Más rápido en M3
    df = df.loc[:, ~df.columns.duplicated()]
    df.to_parquet(csv_file.replace('.csv', '.parquet'), index=False)

In [ ]:
import os
import shutil

# Crear carpetas separadas
os.makedirs("csv_files", exist_ok=True)
os.makedirs("parquet_files", exist_ok=True)

# Mover archivos a sus carpetas
for file in os.listdir():
    if file.endswith(".csv"):
        shutil.move(file, f"csv_files/{file}")
    elif file.endswith(".parquet"):
        shutil.move(file, f"parquet_files/{file}")

In [8]:
from datetime import datetime, time

def parse_hora_flexible(hora_str):
    for fmt in ("%H:%M:%S.%f", "%H:%M:%S"):
        try:
            return datetime.strptime(hora_str, fmt).time()
        except (ValueError, TypeError):
            continue
    return None  


In [10]:
import pandas as pd
from tqdm.notebook import tqdm

# Lista para guardar DataFrames estandarizados
dfs_estandarizados = []

for parquet_file in tqdm(glob.glob("./parquet_files/*.parquet")):
    # 1. Leer el archivo Parquet
    df = pd.read_parquet(parquet_file)

    if "Genero_Usuario" in df.columns:
    # Paso 1: Limpiar y estandarizar (mayúsculas, sin espacios)
        df["Genero_Usuario"] = df["Genero_Usuario"].astype(str).str.upper()
        valores_permitidos = ["M", "F", "O"]
        df["Genero_Usuario"] = df["Genero_Usuario"].where(
        df["Genero_Usuario"].isin(valores_permitidos),
        pd.NA  # Reemplaza valores no permitidos con NA
    )
    
    if "Edad_Usuario" in df.columns:
        df["Edad_Usuario"] = pd.to_numeric(df["Edad_Usuario"], errors="coerce").astype("Int32")
    if "Bici" in df.columns:
        df["Bici"] = pd.to_numeric(df["Bici"], errors="coerce").astype("Int32")
    if "Ciclo_Estacion_Retiro" in df.columns:
        df["Ciclo_Estacion_Retiro"] = pd.to_numeric(df["Ciclo_Estacion_Retiro"], errors="coerce").astype("Int32")
    if "Fecha_Retiro" in df.columns:
        # Si la fecha está como string, la convertimos a datetime
        df["Fecha_Retiro"] = pd.to_datetime(df["Fecha_Retiro"], errors="coerce", dayfirst=True)
    if "Hora_Retiro" in df.columns:
        df["Hora_Retiro"] = df["Hora_Retiro"].apply(parse_hora_flexible)



    if "Ciclo_Estacion_Arribo" in df.columns:
        df["Ciclo_Estacion_Arribo"] = pd.to_numeric(df["Ciclo_Estacion_Arribo"], errors="coerce").astype("Int32")
    if "Fecha_Arribo" in df.columns:
        df["Fecha_Arribo"] = pd.to_datetime(df["Fecha_Arribo"], errors="coerce",dayfirst=True)
    if "Hora_Arribo" in df.columns:
        df["Hora_Arribo"] = df["Hora_Arribo"].apply(parse_hora_flexible)

    
    # 3. Guardar el DataFrame estandarizado
    dfs_estandarizados.append(df)

# 4. Concatenar todos los DataFrames
df_final = pd.concat(dfs_estandarizados, ignore_index=True)

# 5. Guardar el resultado como un nuevo Parquet
df_final.to_parquet("ecobici_final2.parquet", index=False)

  0%|          | 0/183 [00:00<?, ?it/s]

In [11]:
df = pd.read_parquet("ecobici_final2.parquet")
print(df.isnull().sum())  

Genero_Usuario              599336
Edad_Usuario                 12473
Bici                         12690
Ciclo_Estacion_Retiro      1972928
Fecha_Retiro                214222
Hora_Retiro               55485787
Ciclo_Estacion_Arribo      2180404
Fecha_Arribo                214158
Hora_Arribo               56175749
                         120393617
dtype: int64


In [12]:
df = df.drop('', axis=1)


In [13]:
print(len(df))


120393618


In [14]:
print(df.count())


Genero_Usuario           119794282
Edad_Usuario             120381145
Bici                     120380928
Ciclo_Estacion_Retiro    118420690
Fecha_Retiro             120179396
Hora_Retiro               64907831
Ciclo_Estacion_Arribo    118213214
Fecha_Arribo             120179460
Hora_Arribo               64217869
dtype: int64


In [15]:
df.head(5)

,Genero_Usuario,Edad_Usuario,Bici,Ciclo_Estacion_Retiro,Fecha_Retiro,Hora_Retiro,Ciclo_Estacion_Arribo,Fecha_Arribo,Hora_Arribo
0,M,41,5254487,121,2024-08-31,None,134,2024-09-01,None
1,M,25,4809562,135,2024-08-31,None,576,2024-09-01,None
2,M,35,5842372,41,2024-08-31,None,10,2024-09-01,None
3,O,26,7207137,111,2024-08-31,None,136,2024-09-01,None
4,M,20,3196177,295,2024-08-31,None,298,2024-09-01,None


In [ ]:
df.dropna(subset=["Genero_Usuario","Edad_Usuario","Bici", "Fecha_Retiro", "Fecha_Arribo"], inplace=True)


# Para estaciones (categóricas):
df["Ciclo_Estacion_Retiro"].fillna(-1, inplace=True)
df["Ciclo_Estacion_Arribo"].fillna(-1, inplace=True)
df.drop(columns=["Hora_Retiro", "Hora_Arribo"], inplace=True)

In [21]:
print(df.isnull().sum())

Genero_Usuario           0
Edad_Usuario             0
Bici                     0
Ciclo_Estacion_Retiro    0
Fecha_Retiro             0
Ciclo_Estacion_Arribo    0
Fecha_Arribo             0
dtype: int64


In [22]:
# One-Hot para género (ej: "M", "F", "Desconocido")
df = pd.get_dummies(df, columns=["Genero_Usuario"], drop_first=True)

# Ordinal para estaciones (evitar alta dimensionalidad)
from sklearn.preprocessing import OrdinalEncoder
encoder = OrdinalEncoder(handle_unknown="use_encoded_value", unknown_value=-1)
df[["Ciclo_Estacion_Retiro", "Ciclo_Estacion_Arribo"]] = encoder.fit_transform(
    df[["Ciclo_Estacion_Retiro", "Ciclo_Estacion_Arribo"]]
)

In [ ]:
# Convertir fechas y extraer día/mes
df["Dia_Semana_Retiro"] = df["Fecha_Retiro"].dt.dayofweek  # 0=Lunes, 6=Domingo
df["Mes_Retiro"] = df["Fecha_Retiro"].dt.month
df.drop(columns=["Fecha_Retiro", "Fecha_Arribo"], inplace=True)

In [24]:
from sklearn.preprocessing import StandardScaler

scaler = StandardScaler()
df[["Edad_Usuario", "Dia_Semana_Retiro", "Mes_Retiro"]] = scaler.fit_transform(
    df[["Edad_Usuario", "Dia_Semana_Retiro", "Mes_Retiro"]]
)

In [ ]:
df.drop(columns=["Bici"], inplace=True)

In [33]:
df

,Edad_Usuario,Ciclo_Estacion_Retiro,Ciclo_Estacion_Arribo,Genero_Usuario_F,Genero_Usuario_M,Genero_Usuario_O,Dia_Semana_Retiro,Mes_Retiro
0,0.645103,121.0,134.0,False,True,False,1.317134,0.473451
1,-0.940709,135.0,576.0,False,True,False,1.317134,0.473451
2,0.050423,41.0,10.0,False,True,False,1.317134,0.473451
3,-0.841596,111.0,136.0,False,False,True,1.317134,0.473451
4,-1.436275,295.0,298.0,False,True,False,1.317134,0.473451
...,...,...,...,...,...,...,...,...
120393613,-1.138935,74.0,136.0,False,True,False,0.223157,0.473451
120393614,0.149537,73.0,162.0,False,True,False,0.223157,0.473451
120393615,0.744216,128.0,151.0,True,False,False,0.223157,0.473451
120393616,-0.246916,303.0,287.0,False,True,False,0.223157,0.473451


In [31]:
df.to_parquet("datos_procesados.parquet", engine="pyarrow")

In [6]:
import pandas as pd
from sklearn.preprocessing import StandardScaler

# Cargar datos procesados
df2 = pd.read_parquet("datos_procesados.parquet")

# Variables numéricas (ejemplo)
features = ["Edad_Usuario", "Dia_Semana_Retiro", "Mes_Retiro", "Ciclo_Estacion_Retiro", "Ciclo_Estacion_Arribo"]

# Escalar datos
scaler = StandardScaler()
X = scaler.fit_transform(df2[features])

# Aglomerativo

In [7]:
from sklearn.cluster import AgglomerativeClustering

# Definir modelo
agg = AgglomerativeClustering(
    n_clusters=5,          # Número de clusters (ajústalo)
    linkage="ward",        # Método de enlace (óptimo para datos escalados)
    metric="euclidean"     # Métrica de distancia
)

# Entrenar y predecir clusters
df2["cluster_agg"] = agg.fit_predict(X)

MemoryError: Unable to allocate 51.3 PiB for an array with shape (7221201284509878,) and data type float64

# Kmeans

In [ ]:
import dask.dataframe as dd
from dask_ml.preprocessing import StandardScaler
from dask_ml.cluster import KMeans

# Carga CSV o parquet con Dask
df2 = pd.read_parquet("datos_procesados.parquet")

# Selecciona columnas
features = ["Edad_Usuario", "Dia_Semana_Retiro", "Mes_Retiro", "Ciclo_Estacion_Retiro", "Ciclo_Estacion_Arribo"]

# Escalador
scaler = StandardScaler()
X_scaled = scaler.fit_transform(df2)

# KMeans con Dask
kmeans = KMeans(n_clusters=4, random_state=42)
kmeans.fit(X_scaled)

# Puedes acceder a etiquetas
labels = kmeans.labels_.compute()

# GaussianMixture

In [ ]:
from sklearn.mixture import GaussianMixture

# Definir modelo
gmm = GaussianMixture(
    n_components=5,        # Número de clusters
    covariance_type="full",# Tipo de matriz de covarianza ("full", "tied", "diag")
    random_state=42
)

# Entrenar y predecir clusters
df["cluster_gmm"] = gmm.fit_predict(X)